# Lab 16: Data Science Agent avec GCP (BigQuery)## Objectifs- Comprendre l'architecture de l'agent Data Science Google- Explorer NL2SQL et NL2Py pour BigQuery- Integrer BQML pour le machine learning## PrerequisCe lab necessite un projet GCP avec BigQuery API activee.Note: Ce lab presente l'architecture theorique.

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from typing import List, Dict, Optional
from dataclasses import dataclass

from config import get_settings
from utils import LLMClient

In [ ]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

## 2. NL2SQL Translator

In [ ]:
@dataclass
class SQLQuery:
    sql: str
    explanation: str
    tables_used: List[str]

class NL2SQLTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, schema: Dict) -> SQLQuery:
        schema_desc = self._format_schema(schema)
        prompt = f"""Convertis cette question en SQL BigQuery.

SCHEMA:
{schema_desc}

QUESTION: {question}

EXPLANATION: [explication]
SQL:
```sql
[requete]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        sql = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=SQL:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        sql_match = re.search(r'```sql\s*(.*?)\s*```', response, re.DOTALL)
        sql = sql_match.group(1).strip() if sql_match else ''
        return SQLQuery(sql=sql, explanation=explanation, tables_used=[])

    def _format_schema(self, schema: Dict) -> str:
        return '\n'.join([f'Table {t}: {cols}' for t, cols in schema.items()])

## 3. NL2Py Translator

In [ ]:
@dataclass
class PythonCode:
    code: str
    explanation: str

class NL2PyTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, data_context: str) -> PythonCode:
        prompt = f"""Genere du code Python pour: {question}

CONTEXTE: {data_context}

EXPLANATION: [explication]
CODE:
```python
[code]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        code = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=CODE:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        code_match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        code = code_match.group(1).strip() if code_match else ''
        return PythonCode(code=code, explanation=explanation)

## 4. Data Science Agent

In [ ]:
class DataScienceAgent:
    def __init__(self):
        self.llm = LLMClient()
        self.nl2sql = NL2SQLTranslator(self.llm)
        self.nl2py = NL2PyTranslator(self.llm)

    def analyze(self, question: str, schema: Dict, mode: str = 'sql') -> Dict:
        print(f'[AGENT] Question: {question}')
        print(f'[AGENT] Mode: {mode}')
        if mode == 'sql':
            result = self.nl2sql.translate(question, schema)
            return {'type': 'sql', 'query': result.sql, 'explanation': result.explanation}
        elif mode == 'python':
            data_context = self.nl2sql._format_schema(schema)
            result = self.nl2py.translate(question, data_context)
            return {'type': 'python', 'code': result.code, 'explanation': result.explanation}
        return {'error': 'Mode non supporte'}

## 5. Test avec Schema Simule

In [ ]:
schema = {
    'sales': ['date', 'product', 'region', 'quantity', 'revenue'],
    'customers': ['customer_id', 'name', 'segment', 'signup_date'],
    'products': ['product_id', 'name', 'category', 'price']
}

print('Schema BigQuery:')
for table, cols in schema.items():
    print(f'  {table}: {cols}')

In [ ]:
# Test NL2SQL
agent = DataScienceAgent()
question = 'Quel est le revenu total par region?'
result = agent.analyze(question, schema, mode='sql')

print('\\n' + '='*50)
print('RESULTAT NL2SQL:')
print('='*50)
print(f'Explication: {result.get("explanation", "N/A")}')
print(f'SQL: {result.get("query", "N/A")}')

In [ ]:
# Test NL2Py
question2 = 'Calcule la moyenne des revenus par mois'
result2 = agent.analyze(question2, schema, mode='python')

print('\\n' + '='*50)
print('RESULTAT NL2Py:')
print('='*50)
print(f'Explication: {result2.get("explanation", "N/A")}')
print(f'Code: {result2.get("code", "N/A")[:300]}...')

## 6. Resume du Lab### Architecture Google Data Science Agent1. NL2SQL: Question naturelle vers BigQuery2. NL2Py: Question naturelle vers Python3. BQML: Modeles ML dans BigQuery### Prochaine etapeLab 17: Projet Final